# 04. Model Evaluation & SHAP Explainability

Deep dive into model performance metrics, ROC curves, confusion matrices, and SHAP explainability analysis.

In [ ]:
import os, sys, joblib, json
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from src.load_data import ensure_dataset
from src.preprocessing import clean_raw_data, get_transformed_feature_names

model = joblib.load('../models/churn_model.pkl')
pipeline = joblib.load('../models/preprocessing.pkl')
with open('../models/model_meta.json') as f:
    meta = json.load(f)
print(f"Loaded {meta['best_model_name']} with ROC-AUC {meta['metrics']['roc_auc']}")

## 1. SHAP Tree / Linear Explainer Analysis

In [ ]:
df = ensure_dataset()
df_clean = clean_raw_data(df)
X = df_clean.drop(columns=['Churn', 'customerID'], errors='ignore')
X_proc = pipeline.transform(X[:200])
feature_names = meta['feature_names']

explainer = shap.TreeExplainer(model) if hasattr(model, 'feature_importances_') else shap.LinearExplainer(model, X_proc)
shap_values = explainer.shap_values(X_proc)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

shap.summary_plot(shap_values, X_proc, feature_names=feature_names, show=False)
plt.title('Global SHAP Feature Importance Summary')
plt.show()